# Via Array Region Raster

`via_array_region_raster` fills an arbitrary polygon region with a regular grid of via cuts, respecting DRC rules for cut size, enclosure, and spacing.

A common use case is placing vias only where two metal shapes overlap. This notebook shows how to:

1. Define two arbitrary polygon shapes (bottom and top metal).
2. Compute their intersection (logical AND) using Shapely.
3. Pass that intersection region to `via_array_region_raster` to place vias.

## Define two overlapping metal shapes

We define two arbitrary polygons representing the bottom and top metal layers. The via array will only be placed in their overlapping area.

In [ ]:
import gdsfactory as gf
from gdsfactory.gpdk import PDK
from shapely.geometry import Polygon as ShapelyPolygon

gf.gpdk.PDK.activate()

# Bottom metal: a wide rectangle
bottom_metal = ShapelyPolygon([(0, 0), (8, 0), (8, 5), (0, 5)])

# Top metal: a rotated rectangle (diamond-like) that partially overlaps
top_metal = ShapelyPolygon([(2, -1), (10, 2), (8, 6), (0, 3)])

# The via region is the intersection (logical AND) of both shapes
via_region = bottom_metal.intersection(top_metal)

print(f"Bottom metal area: {bottom_metal.area:.2f} um^2")
print(f"Top metal area:    {top_metal.area:.2f} um^2")
print(f"Intersection area: {via_region.area:.2f} um^2")
print(f"Intersection coords: {list(via_region.exterior.coords)}")

## Visualize the overlap

Let's plot the two metal shapes and their intersection to see where the vias will be placed.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon as MplPolygon
from matplotlib.collections import PatchCollection

fig, ax = plt.subplots(1, 1, figsize=(8, 6))

bottom_patch = MplPolygon(list(bottom_metal.exterior.coords), closed=True, alpha=0.3, fc="blue", ec="blue", lw=2, label="Bottom metal (M1)")
top_patch = MplPolygon(list(top_metal.exterior.coords), closed=True, alpha=0.3, fc="red", ec="red", lw=2, label="Top metal (M2)")
intersection_patch = MplPolygon(list(via_region.exterior.coords), closed=True, alpha=0.5, fc="green", ec="green", lw=2, label="Intersection (via region)")

ax.add_patch(bottom_patch)
ax.add_patch(top_patch)
ax.add_patch(intersection_patch)

ax.set_xlim(-1, 11)
ax.set_ylim(-2, 7)
ax.set_aspect("equal")
ax.legend()
ax.set_xlabel("x (um)")
ax.set_ylabel("y (um)")
ax.set_title("Bottom & Top Metal Overlap")
ax.grid(True, alpha=0.3)
plt.show()

## Create the via array

Pass the intersection polygon coordinates to `via_array_region_raster`. The function will:

1. Erode the region inward by `enclosure + half_cut_size` to respect metal enclosure rules.
2. Build a meshgrid of via center candidates with pitch = `cut_size + spacing`.
3. Keep only the centers that fall inside the eroded region.
4. Place via cut polygons at each valid center, plus the full region on both metal layers.

In [ ]:
from gdsfactory.components.vias.via_stack import via_array_region_raster

region_coords = list(via_region.exterior.coords)

c = gf.Component("intersection_via_array")
ref = c.add_ref(via_array_region_raster(
    region=region_coords,
    bottom_layer="M1",
    via_layer="VIA1",
    top_layer="M2",
    via_type="square",
    via_x_minimum_cut_size=0.3,
    via_y_minimum_cut_size=0.3,
    via_x_minimum_enclosure=0.06,
    via_y_minimum_enclosure=0.06,
    via_x_minimum_spacing=0.3,
    via_y_minimum_spacing=0.3,
))

print(f"Cell name: {ref.cell.name}")
print(f"Number of vias placed: {ref.cell.info['num_vias']}")
print(f"Spacing (x, y): ({ref.cell.info['via_x_spacing']}, {ref.cell.info['via_y_spacing']}) um")
print(f"Cut size: {ref.cell.info['via_x_cut_size']} x {ref.cell.info['via_y_cut_size']} um")
c.plot()

## Rectangular region example

For comparison, here is the same function applied to a simple rectangular region.

In [ ]:
c_rect = gf.Component("rectangular_via_array")
ref_rect = c_rect.add_ref(via_array_region_raster(
    region=[(0, 0), (5, 0), (5, 3), (0, 3)],
    bottom_layer="M1",
    via_layer="VIA1",
    top_layer="M2",
    via_x_minimum_cut_size=0.3,
    via_y_minimum_cut_size=0.3,
    via_x_minimum_enclosure=0.06,
    via_y_minimum_enclosure=0.06,
    via_x_minimum_spacing=0.3,
    via_y_minimum_spacing=0.3,
))

print(f"Cell name: {ref_rect.cell.name}")
print(f"Number of vias: {ref_rect.cell.info['num_vias']}")
c_rect.plot()

## L-shaped region example

Non-convex polygons work too. Vias are only placed where the grid centers fall inside the eroded polygon.

In [ ]:
c_L = gf.Component("l_shaped_via_array")
ref_L = c_L.add_ref(via_array_region_raster(
    region=[(0, 0), (5, 0), (5, 2), (3, 2), (3, 4), (0, 4)],
    bottom_layer="M1",
    via_layer="VIA1",
    top_layer="M2",
    via_x_minimum_cut_size=0.3,
    via_y_minimum_cut_size=0.3,
    via_x_minimum_enclosure=0.1,
    via_y_minimum_enclosure=0.1,
    via_x_minimum_spacing=0.3,
    via_y_minimum_spacing=0.3,
))

print(f"Number of vias: {ref_L.cell.info['num_vias']}")
c_L.plot()

## Tighter DRC rules

Increasing the enclosure requirement reduces the number of vias that fit inside the same region.

In [ ]:
c_tight = gf.Component("tight_enclosure_via_array")
ref_tight = c_tight.add_ref(via_array_region_raster(
    region=list(via_region.exterior.coords),
    bottom_layer="M1",
    via_layer="VIA1",
    top_layer="M2",
    via_type="square",
    via_x_minimum_cut_size=0.3,
    via_y_minimum_cut_size=0.3,
    via_x_minimum_enclosure=0.3,
    via_y_minimum_enclosure=0.3,
    via_x_minimum_spacing=0.3,
    via_y_minimum_spacing=0.3,
))

print(f"Number of vias with tight enclosure (0.3 um): {ref_tight.cell.info['num_vias']}")
print(f"vs. loose enclosure (0.06 um): {ref.cell.info['num_vias']}")
c_tight.plot()

In [ ]:
c_tight = gf.Component("tight_enclosure_via_array_rectangle")
ref_tight = c_tight.add_ref(via_array_region_raster(
    region=list(via_region.exterior.coords),
    bottom_layer="M1",
    via_layer="VIA1",
    top_layer="M2",
    via_type="rectangle",
    via_x_minimum_cut_size=0.3,
    via_y_minimum_cut_size=0.3,
    via_x_minimum_enclosure=0.3,
    via_y_minimum_enclosure=0.3,
    via_x_minimum_spacing=0.3,
    via_y_minimum_spacing=0.3,
))

print(f"Number of vias with tight enclosure (0.3 um): {ref_tight.cell.info['num_vias']}")
print(f"vs. loose enclosure (0.06 um): {ref.cell.info['num_vias']}")
c_tight.plot()

## Via stack (multi-layer)

`via_array_stack_oa_compliant` builds a full via stack from `bottom_layer` to `top_layer` by iteratively calling `via_array_region_raster` for each via layer in the connectivity sequence.

The `region`, `size`, and `grid_size` parameters are mutually exclusive (priority: region > size > grid_size). Per-via-layer DRC rules are provided as dictionaries keyed by via layer name.

### Via stack from an arbitrary region (M1 to M3)

In [ ]:
from gdsfactory.components.vias.via_stack import via_array_stack_oa_compliant

c_stack_region = gf.Component("via_stack_region")
ref_stack = c_stack_region.add_ref(via_array_stack_oa_compliant(
    bottom_layer="M1",
    top_layer="M3",
    region=list(via_region.exterior.coords),
    via_type="square",
))

info = ref_stack.cell.info
print(f"Cell name: {ref_stack.cell.name}")
print(f"Stack: {info['bottom_layer']} -> {info['top_layer']}")
for layer_info in info["per_layer_info"]:
    print(f"  {layer_info['via_layer']}: {layer_info['num_vias']} vias "
          f"(cut {layer_info['via_x_cut_size']}x{layer_info['via_y_cut_size']} um, "
          f"spacing {layer_info['via_x_spacing']}x{layer_info['via_y_spacing']} um)")
c_stack_region.plot()

### OpenAccess-compliant metadata

The via stack stores all metadata needed for OA interoperability: layer connectivity, per-layer via counts, cut sizes, spacings, and the enclosing region coordinates.

In [ ]:

info = ref_stack.cell.info

print("=== OpenAccess Via Stack Metadata ===\n")
print(f"  cell_name:                   {ref_stack.cell.name}")
print(f"  bottom_layer:                {info['bottom_layer']}")
print(f"  top_layer:                   {info['top_layer']}")
print(f"  via_type:                    {info['via_type']}")
print(f"  total_num_vias:              {info['total_num_vias']}")
print(f"  num_via_layers:              {info['num_via_layers']}")
print(f"  layer_connectivity_sequence: {info['layer_connectivity_sequence']}")
print(f"  enclosing_region:            {info['enclosing_region']}")

print("\n--- Per-layer info ---")
for i, layer_info in enumerate(info["per_layer_info"]):
    print(f"\n  Via layer {i}: {layer_info['via_layer']} "
          f"({layer_info['bottom_layer']} -> {layer_info['top_layer']})")
    print(f"    num_vias:       {layer_info['num_vias']}")
    print(f"    via_x_cut_size: {layer_info['via_x_cut_size']} um")
    print(f"    via_y_cut_size: {layer_info['via_y_cut_size']} um")
    print(f"    via_x_spacing:  {layer_info['via_x_spacing']} um")
    print(f"    via_y_spacing:  {layer_info['via_y_spacing']} um")

### Via stack with custom per-layer DRC rules

In [ ]:
c_stack_custom = gf.Component("via_stack_custom_rules")
ref_stack_custom = c_stack_custom.add_ref(via_array_stack_oa_compliant(
    bottom_layer="M1",
    top_layer="M3",
    size=(5, 5),
    via_type="square",
    via_x_minimum_cut_size_rules={"VIA": 0.2, "VIA1": 0.25, "VIA2": 0.3},
    via_y_minimum_cut_size_rules={"VIA": 0.2, "VIA1": 0.25, "VIA2": 0.3},
    via_x_minimum_enclosure_rules={"VIA": 0.05, "VIA1": 0.06, "VIA2": 0.08},
    via_y_minimum_enclosure_rules={"VIA": 0.05, "VIA1": 0.06, "VIA2": 0.08},
    via_x_minimum_spacing_rules={"VIA": 0.2, "VIA1": 0.25, "VIA2": 0.3},
    via_y_minimum_spacing_rules={"VIA": 0.2, "VIA1": 0.25, "VIA2": 0.3},
))

info = ref_stack_custom.cell.info
print(f"Stack: {info['bottom_layer']} -> {info['top_layer']}")
for layer_info in info["per_layer_info"]:
    print(f"  {layer_info['via_layer']}: {layer_info['num_vias']} vias "
          f"(cut {layer_info['via_x_cut_size']}x{layer_info['via_y_cut_size']} um)")
c_stack_custom.plot()

In [ ]:
c_stack_custom = gf.Component("via_stack_custom_rules_rectangle")
ref_stack_custom = c_stack_custom.add_ref(via_array_stack_oa_compliant(
    bottom_layer="M1",
    top_layer="M3",
    size=(5, 5),
    via_type="rectangle",
    via_x_minimum_cut_size_rules={"VIA": 0.2, "VIA1": 0.25, "VIA2": 0.3},
    via_y_minimum_cut_size_rules={"VIA": 0.2, "VIA1": 0.25, "VIA2": 0.3},
    via_x_minimum_enclosure_rules={"VIA": 0.05, "VIA1": 0.06, "VIA2": 0.08},
    via_y_minimum_enclosure_rules={"VIA": 0.05, "VIA1": 0.06, "VIA2": 0.08},
    via_x_minimum_spacing_rules={"VIA": 0.2, "VIA1": 0.25, "VIA2": 0.3},
    via_y_minimum_spacing_rules={"VIA": 0.2, "VIA1": 0.25, "VIA2": 0.3},
))

info = ref_stack_custom.cell.info
print(f"Stack: {info['bottom_layer']} -> {info['top_layer']}")
for layer_info in info["per_layer_info"]:
    print(f"  {layer_info['via_layer']}: {layer_info['num_vias']} vias "
          f"(cut {layer_info['via_x_cut_size']}x{layer_info['via_y_cut_size']} um)")
c_stack_custom.plot()

In [ ]:
c_stack_region = gf.Component("via_stack_region_rectangle")
ref_stack = c_stack_region.add_ref(via_array_stack_oa_compliant(
    bottom_layer="M1",
    top_layer="M3",
    region=list(via_region.exterior.coords),
    via_type="rectangle",
))

info = ref_stack.cell.info
print(f"Stack: {info['bottom_layer']} -> {info['top_layer']}")
for layer_info in info["per_layer_info"]:
    print(f"  {layer_info['via_layer']}: {layer_info['num_vias']} vias "
          f"(cut {layer_info['via_x_cut_size']}x{layer_info['via_y_cut_size']} um, "
          f"spacing {layer_info['via_x_spacing']}x{layer_info['via_y_spacing']} um)")
c_stack_region.plot()

In [ ]:
c_stack_region = gf.Component("m1_m2_via_stack_region_rectangle")
ref_stack = c_stack_region.add_ref(via_array_stack_oa_compliant(
    bottom_layer="M1",
    top_layer="M2",
    region=list(via_region.exterior.coords),
    via_type="rectangle",
))

info = ref_stack.cell.info
print(f"Cell name: {ref_stack.cell.name}")
print(f"Stack: {info['bottom_layer']} -> {info['top_layer']}")
for layer_info in info["per_layer_info"]:
    print(f"  {layer_info['via_layer']}: {layer_info['num_vias']} vias "
          f"(cut {layer_info['via_x_cut_size']}x{layer_info['via_y_cut_size']} um, "
          f"spacing {layer_info['via_x_spacing']}x{layer_info['via_y_spacing']} um)")
c_stack_region.plot()